In [1]:
from HINGE.evaluation_synthetic import run_synthetic_evaluation_with_dropout

from HINGE.feature_mapping import prepare_expression_features

from sklearn.preprocessing import normalize
import pickle
from pathlib import Path

import matplotlib.pyplot as plt

In [2]:
SAVE_DIR = Path('./hinge_checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
n_gene_comps = 128

In [5]:
with open(SAVE_DIR / 'checkpoint_step1.pkl', "rb") as f:
    data_dict=pickle.load(f)

data_dict

{'RNA_panel1': {'s1': AnnData object with n_obs × n_vars = 6989 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's2': AnnData object with n_obs × n_vars = 8978 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's3': AnnData object with n_obs × n_vars = 449221 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's4': AnnData object with n_obs × n_vars = 7442 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's5': AnnData object with n_obs × n_vars = 16540 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'f

In [6]:
#select which sample to use for the mapping

uni_480_list = [data_dict['HE_panel1'][1]]  # only the first sample for demonstration
uni_5k_list = [data_dict['HE_panel2'][1]]  # only the first sample for demonstration
rna_480_list = [data_dict['RNA_panel1']['s2']]  # only the first sample for demonstration
rna_5k_list = [data_dict['RNA_panel2']['s8']]  # only the first sample for demonstration

In [7]:
uni_480_list

[AnnData object with n_obs × n_vars = 8978 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [8]:

uni_480_normalized_list = []

for adata in uni_480_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_480_normalized_list.append(adata_norm)

In [9]:
uni_5k_normalized_list = []

for adata in uni_5k_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_5k_normalized_list.append(adata_norm)

In [10]:
uni_480_normalized_list

[AnnData object with n_obs × n_vars = 8978 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [11]:
adata_panel1          = rna_480_list[0]
uni_panel1_normed     = uni_480_normalized_list[0]
adata_panel2          = rna_5k_list[0]
uni_panel2_normed     = uni_5k_normalized_list[0]

# run test

In [12]:
df_dropout, fakes, stats = run_synthetic_evaluation_with_dropout(
    adata_panel1          = adata_panel1,
    uni_panel1_normed     = uni_panel1_normed,
    adata_panel2          = adata_panel2,
    uni_panel2_normed     = uni_panel2_normed,
    alpha_list            = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5],
    n_seeds_per_alpha     = 2,
    MAX_DIST              = None,
    dropout_frac          = 0.10,
    dropout_mode          = "independent",   # or "symmetric"
    SAMPLE_NAME           = "s1",
    evaluate_hinge = True,
    
    leiden_resolution     = 0.5,         # tune if needed
    min_cells_per_cluster = 20,
)

panel1 shape: (8978, 5101)
panel2 shape: (8899, 480)
  Computing population-level difference statistics
  Panel1: 8978 cells  |  Panel2: 8899 cells

  Coordinate displacement field:
    median dx        = -0.68 um
    median dy        = -0.19 um
    median NN dist   = 22.87 um
    kept fraction    = 100.0%

  UNI (normalized space, 2048D):
    |delta_mean| median = 0.00264
    delta_std    median = 0.01713

  Expression (444 genes):
    |delta_mean| median = 0.0622
    delta_std    median = 0.1445
  Created s1_t: alpha=0.25  seed=0  xy_method=displacement_field  mean coord shift=12.96 um  n=8978
  Created s1_t: alpha=0.25  seed=1  xy_method=displacement_field  mean coord shift=12.97 um  n=8978
  Created s1_t: alpha=0.50  seed=2  xy_method=displacement_field  mean coord shift=21.37 um  n=8978
  Created s1_t: alpha=0.50  seed=3  xy_method=displacement_field  mean coord shift=21.29 um  n=8978
  Created s1_t: alpha=0.75  seed=4  xy_method=displacement_field  mean coord shift=30.71 um  n=89

/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:75: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_pca.py:385: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)
/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yyamamo2/lupus_project/HINGE-Histology-and-gene-guided-cell-matching-ensemble-main/HINGE/gene_agreement.py:412: FutureWarning: In the future, the 


  Panel 1 Leiden clustering  (resolution=0.5, full gene set)
    Clusters         : 16
    Cluster sizes    : min=112, median=532, max=792

  Label transfer to Panel 2  (KNN k=15, metric=cosine)
    Cluster       P1 frac    P2 frac
    0               0.098      0.091
    1               0.096      0.083
    10              0.048      0.043
    11              0.035      0.023
    12              0.030      0.030
    13              0.028      0.030
    14              0.015      0.016
    15              0.014      0.015
    2               0.094      0.108
    3               0.093      0.134
    4               0.093      0.077
    5               0.092      0.101
    6               0.079      0.083
    7               0.069      0.045
    8               0.063      0.064
    9               0.054      0.058
    (large P1/P2 fraction differences may indicate true composition differences between sections)

    Valid clusters   : 16 / 16  (≥20 cells per panel)

    Structure scores 

/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:75: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_pca.py:385: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)



  Panel 1 Leiden clustering  (resolution=0.5, full gene set)
    Clusters         : 16
    Cluster sizes    : min=8, median=529, max=912

  Label transfer to Panel 2  (KNN k=15, metric=cosine)
    Cluster       P1 frac    P2 frac
    0               0.113      0.160
    1               0.108      0.094
    10              0.045      0.036
    11              0.045      0.040
    12              0.034      0.022
    13              0.031      0.031
    14              0.015      0.016
    15              0.001      0.001
    2               0.092      0.096
    3               0.085      0.100
    4               0.084      0.093
    5               0.083      0.064
    6               0.074      0.076
    7               0.068      0.068
    8               0.063      0.063
    9               0.059      0.039
    (large P1/P2 fraction differences may indicate true composition differences between sections)

    Valid clusters   : 15 / 16  (≥20 cells per panel)

    Structure scores (r

/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:75: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_pca.py:385: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)


... (15 outputs left out)

/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:75: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_pca.py:385: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)



  Panel 1 Leiden clustering  (resolution=0.5, full gene set)
    Clusters         : 16
    Cluster sizes    : min=7, median=463, max=1026

  Label transfer to Panel 2  (KNN k=15, metric=cosine)
    Cluster       P1 frac    P2 frac
    0               0.127      0.216
    1               0.102      0.009
    10              0.054      0.025
    11              0.046      0.044
    12              0.032      0.017
    13              0.022      0.018
    14              0.015      0.050
    15              0.001      0.000
    2               0.101      0.335  ⚠️ 
    3               0.095      0.017
    4               0.094      0.077
    5               0.073      0.035
    6               0.068      0.058
    7               0.059      0.008
    8               0.055      0.021
    9               0.055      0.067
    (large P1/P2 fraction differences may indicate true composition differences between sections)

    Valid clusters   : 15 / 16  (≥20 cells per panel)

    Structure sco

/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:75: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/scanpy/preprocessing/_pca.py:385: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)



  Panel 1 Leiden clustering  (resolution=0.5, full gene set)
    Clusters         : 16
    Cluster sizes    : min=109, median=530, max=1036

  Label transfer to Panel 2  (KNN k=15, metric=cosine)
    Cluster       P1 frac    P2 frac
    0               0.128      0.016
    1               0.113      0.078
    10              0.031      0.014
    11              0.030      0.020
    12              0.019      0.066
    13              0.014      0.048
    14              0.014      0.014
    15              0.013      0.001
    2               0.109      0.409  ⚠️ 
    3               0.102      0.068
    4               0.083      0.017
    5               0.081      0.090
    6               0.073      0.010
    7               0.072      0.005
    8               0.060      0.023
    9               0.058      0.122
    (large P1/P2 fraction differences may indicate true composition differences between sections)

    Valid clusters   : 15 / 16  (≥20 cells per panel)

    Structure s

In [13]:
import sys
import importlib.util

print("Python:", sys.executable)
print("Version:", sys.version)

print("igraph:", importlib.util.find_spec("igraph"))
print("leidenalg:", importlib.util.find_spec("leidenalg"))

Python: /home/yyamamo2/.conda/envs/cosie_env/bin/python
Version: 3.9.19 (main, May  6 2024, 19:43:03) 
[GCC 11.2.0]
igraph: ModuleSpec(name='igraph', loader=<_frozen_importlib_external.SourceFileLoader object at 0x14a4c3c02910>, origin='/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/igraph/__init__.py', submodule_search_locations=['/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/igraph'])
leidenalg: ModuleSpec(name='leidenalg', loader=<_frozen_importlib_external.SourceFileLoader object at 0x14a4c3bf6c10>, origin='/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/leidenalg/__init__.py', submodule_search_locations=['/home/yyamamo2/.conda/envs/cosie_env/lib/python3.9/site-packages/leidenalg'])
